# Spike B — CSM-1B smoke + pt-BR (Colab)

Decides the bet with numbers: latency/RTF, pt-BR WER (no fine-tune), in-context clone.

**Before running:** Runtime > Change runtime type > GPU (T4/L4/A100). You need a Hugging Face account with **accepted licenses** for `meta-llama/Llama-3.2-1B` (gated) **and** `sesame/csm-1b`.

This notebook expects the TTS-ptbr repo present (clone your private repo in Colab, or upload the `phase0/spike_b_csm/` scripts).

In [ ]:
# 1. Get the code: clone the (public) Sesame CSM repo + ensure spike scripts are here.
import os
!git clone --depth 1 https://github.com/SesameAILabs/csm.git /content/csm
# If running outside the TTS-ptbr repo, clone it (private -> needs a token):
#   !git clone https://<TOKEN>@github.com/pedrocormann/TTS-ptbr.git /content/TTS-ptbr
# then: %cd /content/TTS-ptbr/phase0/spike_b_csm
print('csm at /content/csm ; run the next cells from phase0/spike_b_csm/')

In [ ]:
# 2. Install the CSM-pinned deps + spike extras (faster-whisper, jiwer).
!pip -q install -r requirements.txt
import os; os.environ['NO_TORCH_COMPILE'] = '1'  # required by Mimi

In [ ]:
# 3. Hugging Face login. You MUST have accepted BOTH model licenses on hf.co first:
#    huggingface.co/meta-llama/Llama-3.2-1B  (gated, click 'agree')
#    huggingface.co/sesame/csm-1b
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# 4. THE smoke test — prints the Spike B go/no-go table (latency, RTF, pt-BR WER).
!python smoke_csm.py --csm-repo /content/csm --out-dir out
# Listen to the pt-BR sample (numbers don't capture prosody):
import IPython.display as ipd; ipd.Audio('out/ptbr_nofinetune.wav')

In [ ]:
# 5. (optional) pt-BR data prep + QLoRA SCAFFOLD dry-run (not turnkey — see qlora_finetune.py).
!python prep_ptbr_data.py --download --out-dir data_ptbr
!python qlora_finetune.py --csm-repo /content/csm --manifest data_ptbr/manifest.jsonl --dry-run

Record latency/RTF + pt-BR WER in `../../specs/tech-stack.md` and the Dev KB, then run `/sdd` ATUALIZAR + `/feature-spec` Phase 1. The QLoRA cell intentionally stops at a scaffold boundary.